In [0]:
# Databricks notebook source
# Project: Azure Databricks Retail Supply Chain Lakehouse
# Notebook: 00-setup
# Purpose: Create the Unity Catalog structure and external volumes.

STORAGE_ACCOUNT = "stretailscmfaye01"
CATALOG_NAME = "retail_dev"

MANAGED_LOCATION = (
    f"abfss://managed@{STORAGE_ACCOUNT}"
    ".dfs.core.windows.net/retail-dev"
)

print(f"Storage account: {STORAGE_ACCOUNT}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Managed location: {MANAGED_LOCATION}")

In [0]:
# Create the project catalog

spark.sql(
    f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}
    MANAGED LOCATION '{MANAGED_LOCATION}'
    """
)

print(f"Catalog {CATALOG_NAME} created successfully.")

In [0]:
# Create the Medallion Architecture schemas

schemas = {
    "bronze": "Raw data ingested from source systems",
    "silver": "Cleaned, validated and deduplicated data",
    "gold": "Business-ready tables and analytical KPIs",
    "ops": "Technical volumes, checkpoints and ingestion metadata"
}

for schema_name, description in schemas.items():
    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{schema_name}
        COMMENT '{description}'
        """
    )

    print(f"Schema created: {CATALOG_NAME}.{schema_name}")

In [0]:
# Create external volumes for files managed outside Delta tables

volumes = {
    "landing_volume": (
        f"abfss://landing@{STORAGE_ACCOUNT}"
        ".dfs.core.windows.net/"
    ),
    "technical_volume": (
        f"abfss://technical@{STORAGE_ACCOUNT}"
        ".dfs.core.windows.net/"
    ),
    "quarantine_volume": (
        f"abfss://quarantine@{STORAGE_ACCOUNT}"
        ".dfs.core.windows.net/"
    )
}

for volume_name, location in volumes.items():
    spark.sql(
        f"""
        CREATE EXTERNAL VOLUME IF NOT EXISTS
        {CATALOG_NAME}.ops.{volume_name}
        LOCATION '{location}'
        """
    )

    print(f"Volume created: {CATALOG_NAME}.ops.{volume_name}")

In [0]:
print("Schemas:")
display(
    spark.sql(
        f"SHOW SCHEMAS IN {CATALOG_NAME}"
    )
)

print("Volumes:")
display(
    spark.sql(
        f"SHOW VOLUMES IN {CATALOG_NAME}.ops"
    )
)

In [0]:
test_path = (
    "/Volumes/retail_dev/ops/"
    "landing_volume/connection-test"
)

test_df = spark.createDataFrame(
    [
        (
            1,
            "Databricks successfully connected to ADLS Gen2"
        )
    ],
    ["test_id", "message"]
)

(
    test_df.write
    .format("json")
    .mode("overwrite")
    .save(test_path)
)

display(
    spark.read
    .format("json")
    .load(test_path)
)

print("ADLS connection test completed successfully.")